In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/preprocessed_data.csv")

df.head()

,CustomerID,ProductID,Quantity,Price,TransactionDate,PaymentMethod,StoreLocation,ProductCategory,DiscountApplied(%),TotalAmount,Year,Month,Day,Week,Quarter,DayOfWeek,Weekend,Season
0,109318,C,7,80.079844,2023-12-26 12:32:00,Cash,"176 Andrew Cliffs\nBaileyfort, HI 93354",Books,18.677100,455.862764,2023,12,26,52,4,Tuesday,False,Winter
1,993229,C,4,75.195229,2023-08-05 00:00:00,Cash,"11635 William Well Suite 809\nEast Kara, MT 19483",Home Decor,14.121365,258.306546,2023,8,5,31,3,Saturday,True,Monsoon
2,579675,A,8,31.528816,2024-03-11 18:51:00,Cash,"910 Mendez Ville Suite 909\nPort Lauraland, MO...",Books,15.943701,212.015651,2024,3,11,11,1,Monday,False,Summer
3,799826,D,5,98.880218,2023-10-27 22:00:00,PayPal,"87522 Sharon Corners Suite 500\nLake Tammy, MO...",Books,6.686337,461.343769,2023,10,27,43,4,Friday,False,Autumn
4,121413,A,7,93.188512,2023-12-22 11:38:00,Cash,"0070 Michelle Island Suite 143\nHoland, VA 80142",Electronics,4.030096,626.030484,2023,12,22,51,4,Friday,False,Winter


In [3]:
df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])

In [4]:
latest_date = df["TransactionDate"].max()

customer_churn = df.groupby("CustomerID").agg({
    "TransactionDate": "max",
    "TotalAmount": "sum",
    "Quantity": "sum"
}).reset_index()

customer_churn.rename(columns={
    "TransactionDate": "LastPurchaseDate",
    "TotalAmount": "TotalSpent",
    "Quantity": "TotalQuantity"
}, inplace=True)

customer_churn.head()

,CustomerID,LastPurchaseDate,TotalSpent,TotalQuantity
0,14,2023-08-06 06:45:00,256.232791,5
1,42,2023-05-19 21:52:00,502.656523,7
2,49,2023-06-05 13:10:00,21.399047,1
3,59,2024-04-01 01:06:00,249.492696,12
4,65,2023-06-18 10:29:00,548.006625,8


In [5]:
customer_churn["InactiveDays"] = (
    latest_date - customer_churn["LastPurchaseDate"]
).dt.days

customer_churn.head()

,CustomerID,LastPurchaseDate,TotalSpent,TotalQuantity,InactiveDays
0,14,2023-08-06 06:45:00,256.232791,5,266
1,42,2023-05-19 21:52:00,502.656523,7,345
2,49,2023-06-05 13:10:00,21.399047,1,328
3,59,2024-04-01 01:06:00,249.492696,12,27
4,65,2023-06-18 10:29:00,548.006625,8,315


In [6]:
customer_churn["ChurnRisk"] = np.where(
    customer_churn["InactiveDays"] > 60,
    "At Risk",
    "Not At Risk"
)

customer_churn.head()

,CustomerID,LastPurchaseDate,TotalSpent,TotalQuantity,InactiveDays,ChurnRisk
0,14,2023-08-06 06:45:00,256.232791,5,266,At Risk
1,42,2023-05-19 21:52:00,502.656523,7,345,At Risk
2,49,2023-06-05 13:10:00,21.399047,1,328,At Risk
3,59,2024-04-01 01:06:00,249.492696,12,27,Not At Risk
4,65,2023-06-18 10:29:00,548.006625,8,315,At Risk


In [7]:
customer_churn["ChurnRisk"].value_counts()

ChurnRisk
At Risk        78697
Not At Risk    16518
Name: count, dtype: int64

In [8]:
customer_churn.sort_values("InactiveDays", ascending=False).head(10)

,CustomerID,LastPurchaseDate,TotalSpent,TotalQuantity,InactiveDays,ChurnRisk
9378,99136,2023-04-30 18:16:00,135.334525,2,364,At Risk
95026,997878,2023-04-30 14:04:00,148.315112,5,364,At Risk
9493,100418,2023-04-30 18:16:00,99.592033,3,364,At Risk
79997,840208,2023-04-30 05:06:00,98.315023,3,364,At Risk
79909,839280,2023-04-30 17:06:00,158.692489,3,364,At Risk
80836,848713,2023-04-30 08:17:00,96.333440,2,364,At Risk
94859,996116,2023-04-30 07:44:00,648.314153,9,364,At Risk
9888,104602,2023-04-30 17:27:00,207.116882,5,364,At Risk
79760,837812,2023-04-30 18:52:00,116.356112,6,364,At Risk
30931,324640,2023-04-30 07:11:00,573.383871,8,364,At Risk


In [9]:
customer_churn.sort_values("InactiveDays").head(10)

,CustomerID,LastPurchaseDate,TotalSpent,TotalQuantity,InactiveDays,ChurnRisk
47911,502736,2024-04-28 16:12:00,100.028601,5,0,Not At Risk
92569,971943,2024-04-28 03:36:00,443.523592,8,0,Not At Risk
82592,866543,2024-04-28 10:17:00,778.106168,13,0,Not At Risk
67782,712533,2024-04-28 21:12:00,351.282645,5,0,Not At Risk
56375,592253,2024-04-28 19:27:00,311.519236,5,0,Not At Risk
49011,513982,2024-04-28 22:03:00,46.844668,3,0,Not At Risk
90549,950501,2024-04-28 06:41:00,76.850478,6,0,Not At Risk
82526,865897,2024-04-28 12:35:00,269.834323,3,0,Not At Risk
58541,615272,2024-04-28 21:31:00,164.270991,7,0,Not At Risk
26467,278146,2024-04-28 11:33:00,338.689209,8,0,Not At Risk


In [10]:
customer_churn.to_csv(
    "../data/customer_churn.csv",
    index=False
)

print("Customer Churn Data Saved Successfully!")

Customer Churn Data Saved Successfully!
